In [1]:
import geopandas as gpd
import rasterio
import rasterio.mask
import pandas as pd
import numpy as np
import geopandas as gpd
from shapely.geometry import Point

In [2]:
url = "https://naturalearth.s3.amazonaws.com/110m_cultural/ne_110m_admin_0_countries.zip"

world = gpd.read_file(url)

In [3]:
germany = world[world["ADMIN"] == "Germany"]

In [4]:
src = rasterio.open("JRC-ESTAT_Census_Population_2021_100m.tif")

germany = germany.to_crs(src.crs)

In [5]:
out_image, out_transform = rasterio.mask.mask(
    src,
    germany.geometry,
    crop=True
)

In [6]:
rows, cols = np.where(out_image[0] > 0)

values = out_image[0][rows, cols]

xs, ys = rasterio.transform.xy(out_transform, rows, cols)

In [7]:
df = pd.DataFrame({
    "x": xs,
    "y": ys,
    "population": values
})

In [ ]:
import geopandas as gpd
from shapely.geometry import Point

df["geometry"] = df.apply(lambda r: Point(r["x"], r["y"]), axis=1)

gdf = gpd.GeoDataFrame(df, geometry="geometry", crs="EPSG:3035")

gdf = gdf.to_crs(epsg=4326)

In [9]:
gdf.to_csv("../processed/density.csv",
                index=False,
                encoding="utf-8")